# Conservative single-grain geometry generation

Generate and inspect conservative adaptive Cartesian cube fills for supported analytic domains. The adaptive meshes contain only axis-aligned cubes that are provably fully inside the target domain; ambiguous boundary cells are refined or left unresolved, never accepted.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from math import sqrt
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / 'utils').is_dir():
    SINGLE_GRAIN_DIR = Path('python/experiments/single_grain').resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from utils.geometry import (
    AxisAlignedBox,
    Cylinder,
    Ellipsoid,
    H0_REFERENCE_TABLE,
    HexagonalPrism,
    Sphere,
    adaptive_cube_fill,
    assert_no_cube_overlaps,
    calibrate_h0_sweep,
    estimate_h0_from_volume,
    fill_domain_mesh,
    generate_cylinder,
    generate_ellipsoid,
    generate_hexagonal_prism,
    generate_rectangular_prism,
    generate_sphere,
    recommended_h0,
    select_best_h0_records,
)
from utils.geometry_plotting import interactive_prism_mesh, plot_prism_mesh

plt.rcParams.update({'figure.dpi': 110})


## Configuration

All dimensions are in metres. `TARGET_TILES` must be one of the calibrated target counts `2^8` through `2^12`. The adaptive fills use a relative `h0` lookup measured in `calibrate_h0_resolution.ipynb`, so the starting tile size scales with each shape's characteristic length instead of an absolute particle size.

The supported calibration cases in this notebook are: sphere, cylinder with `H/R = 2`, ellipsoid with `b/a = 0.6` and `c/a = 0.4`, and hexagonal prism with `H/side = 1`. If `TARGET_TILES` is outside the calibrated set, the lookup raises `ValueError("best h0 not calibrated for the target number of tiles")`.

`EPS` is the conservative geometric tolerance in metres. `H_MIN_RATIO` sets the finest allowed boundary cube size relative to the calibrated `h0`, and `MAX_DEPTH` is the maximum octree level. Complete-layer filling is the default, so target acceptance is checked only after full refinement layers.


In [ ]:
NM = 1e-9
TARGET_TILES = 2**12
EPS = 1e-18
MAX_DEPTH = 8
H_MIN_RATIO = 1 / 16
TARGET_LOWER_TOLERANCE = 0.10

CALIBRATED_RELATIVE_H0 = {
    "cylinder_hr_1": {
        "1024": 0.125418229343,
        "2048": 0.141457994428,
        "256": 0.199089029195,
        "4096": 0.112275284582,
        "512": 0.257817320151
    },
    "cylinder_hr_2": {
        "1024": 0.15801706719,
        "2048": 0.178225904856,
        "256": 0.409259485224,
        "4096": 0.141457994428,
        "512": 0.199089029195
    },
    "cylinder_hr_4": {
        "1024": 0.199089029195,
        "2048": 0.224550569164,
        "256": 0.515634640303,
        "4096": 0.178225904856,
        "512": 0.409259485224
    },
    "ellipsoid_ba_0.5_ca_0.25": {
        "1024": 0.069020311496,
        "2048": 0.077847334396,
        "256": 0.17876054565,
        "4096": 0.061787470257,
        "512": 0.141882339108
    },
    "ellipsoid_ba_0.6_ca_0.4": {
        "1024": 0.08578486218,
        "2048": 0.096755907175,
        "256": 0.22218023129,
        "4096": 0.076795214417,
        "512": 0.108082153623
    },
    "ellipsoid_ba_0.8_ca_0.6": {
        "1024": 0.108082153623,
        "2048": 0.121904804151,
        "256": 0.279929550273,
        "4096": 0.096755907175,
        "512": 0.136174980468
    },
    "hex_hs_0.5": {
        "1024": 0.093436810336,
        "2048": 0.07416084551,
        "256": 0.132708881439,
        "4096": 0.058861502089,
        "512": 0.105331109001
    },
    "hex_hs_1": {
        "1024": 0.117723004177,
        "2048": 0.093436810336,
        "256": 0.167202713233,
        "4096": 0.0819672503,
        "512": 0.210772929344
    },
    "hex_hs_2": {
        "1024": 0.210772929344,
        "2048": 0.167290584884,
        "256": 0.384148750473,
        "4096": 0.093436810336,
        "512": 0.304899065307
    },
    "sphere": {
        "1024": 0.138040622992,
        "2048": 0.155694668792,
        "256": 0.196059953293,
        "4096": 0.123574940513,
        "512": 0.173920286648
    }
}
CALIBRATED_TARGET_TILES = tuple(2**power for power in range(8, 13))
CALIBRATION_ERROR = 'best h0 not calibrated for the target number of tiles'

def calibrated_relative_h0(shape_key, target):
    target = int(target)
    if target not in CALIBRATED_TARGET_TILES:
        raise ValueError(CALIBRATION_ERROR)
    try:
        return CALIBRATED_RELATIVE_H0[shape_key][str(target)]
    except KeyError as exc:
        raise ValueError(f'best h0 not calibrated for shape {shape_key!r}') from exc

def calibrated_h0_for_target(shape_key, scale_length, target):
    return calibrated_relative_h0(shape_key, target) * float(scale_length)

def h_min_from_h0(h0):
    return float(h0) * H_MIN_RATIO

def coarse_h0_for_target(shape_key, scale_length, target):
    return calibrated_h0_for_target(shape_key, scale_length, target)

def h_min_for_target(shape_key, scale_length, target):
    return h_min_from_h0(coarse_h0_for_target(shape_key, scale_length, target))

def coarse_h0(shape_key, scale_length):
    return coarse_h0_for_target(shape_key, scale_length, TARGET_TILES)

def h_min_for(shape_key, scale_length):
    return h_min_for_target(shape_key, scale_length, TARGET_TILES)

QUEUE_POLICY = 'symmetric_priority'

BOX_DIMENSIONS = (80 * NM, 60 * NM, 40 * NM)
BOX_RESOLUTION = (8, 6, 4)
SPHERE_RADIUS = 40 * NM
ELLIPSOID_MAJOR_AXIS = 50 * NM
ELLIPSOID_SEMI_AXES = (ELLIPSOID_MAJOR_AXIS, 0.6 * ELLIPSOID_MAJOR_AXIS, 0.4 * ELLIPSOID_MAJOR_AXIS)
CYLINDER_RADIUS = 30 * NM
CYLINDER_LENGTH = 2.0 * CYLINDER_RADIUS
CYLINDER_AXIS = 'z'
HEXAGON_SIDE_LENGTH = 40 * NM
HEXAGON_HEIGHT = 1.0 * HEXAGON_SIDE_LENGTH
HEXAGON_AXIS = 'z'
HEXAGON_ROTATION_DEGREES = 0.0
ROTATED_HEXAGON_DEGREES = 15.0

SPHERE_H0_KEY = 'sphere'
ELLIPSOID_H0_KEY = 'ellipsoid_ba_0.6_ca_0.4'
CYLINDER_H0_KEY = 'cylinder_hr_2'
HEXAGON_H0_KEY = 'hex_hs_1'
ROTATED_HEXAGON_H0_KEY = HEXAGON_H0_KEY

SPHERE_H0_SCALE = SPHERE_RADIUS
ELLIPSOID_H0_SCALE = ELLIPSOID_MAJOR_AXIS
CYLINDER_H0_SCALE = CYLINDER_RADIUS
HEXAGON_H0_SCALE = HEXAGON_SIDE_LENGTH
ROTATED_HEXAGON_H0_SCALE = HEXAGON_SIDE_LENGTH


## Which API should I use?

- `Sphere(...)`, `Cylinder(...)`, `HexagonalPrism(...)`, etc. create reusable domain-oracle objects. They know the shape, volume, bounding box, signed distance, and conservative cube classifier.
- `adaptive_cube_fill(domain, ...)` is the low-level algorithm. It returns raw `Cube` objects plus `CubeFillStats`.
- `fill_domain_mesh(domain, ...)` is the generic MagTense adapter. It runs `adaptive_cube_fill` for any domain oracle and returns a `PrismMesh` with `centers`, `dimensions`, `levels`, and metadata.
- `generate_sphere(...)`, `generate_cylinder(...)`, `generate_hexagonal_prism(...)`, etc. are convenience shortcuts. They construct the domain oracle for you and then call `fill_domain_mesh`.

Refinement scheduling is controlled by `queue_policy`. The default `'breadth_first'` is the simplest traversal. This notebook uses `'symmetric_priority'`, which refines mirrored boundary cells as a batch and favors radial/off-axis boundary cells for cylinders and hexagonal prisms. That gives a more balanced-looking adaptive mesh while keeping the same conservative inside tests.


## Generate the meshes


In [ ]:
box = generate_rectangular_prism(
    BOX_DIMENSIONS, *BOX_RESOLUTION,
)

sphere_domain = Sphere(center=[0.0, 0.0, 0.0], radius=SPHERE_RADIUS)
ellipsoid_domain = Ellipsoid(center=[0.0, 0.0, 0.0], semi_axes=ELLIPSOID_SEMI_AXES)
cylinder_domain = Cylinder(
    center=[0.0, 0.0, 0.0],
    radius=CYLINDER_RADIUS,
    length=CYLINDER_LENGTH,
    axis=CYLINDER_AXIS,
)
hexagon_domain = HexagonalPrism(
    center=[0.0, 0.0, 0.0],
    side_length=HEXAGON_SIDE_LENGTH,
    height=HEXAGON_HEIGHT,
    axis=HEXAGON_AXIS,
    rotation_degrees=HEXAGON_ROTATION_DEGREES,
)
rotated_hexagon_domain = HexagonalPrism(
    center=[0.0, 0.0, 0.0],
    side_length=HEXAGON_SIDE_LENGTH,
    height=HEXAGON_HEIGHT,
    axis=HEXAGON_AXIS,
    rotation_degrees=ROTATED_HEXAGON_DEGREES,
)
box_domain = AxisAlignedBox(center=[0.0, 0.0, 0.0], dimensions=BOX_DIMENSIONS)

sphere_h0 = coarse_h0(SPHERE_H0_KEY, SPHERE_H0_SCALE)
sphere_h_min = h_min_for(SPHERE_H0_KEY, SPHERE_H0_SCALE)
ellipsoid_h0 = coarse_h0(ELLIPSOID_H0_KEY, ELLIPSOID_H0_SCALE)
ellipsoid_h_min = h_min_for(ELLIPSOID_H0_KEY, ELLIPSOID_H0_SCALE)
cylinder_h0 = coarse_h0(CYLINDER_H0_KEY, CYLINDER_H0_SCALE)
cylinder_h_min = h_min_for(CYLINDER_H0_KEY, CYLINDER_H0_SCALE)
hexagon_h0 = coarse_h0(HEXAGON_H0_KEY, HEXAGON_H0_SCALE)
hexagon_h_min = h_min_for(HEXAGON_H0_KEY, HEXAGON_H0_SCALE)
rotated_hexagon_h0 = coarse_h0(ROTATED_HEXAGON_H0_KEY, ROTATED_HEXAGON_H0_SCALE)
rotated_hexagon_h_min = h_min_for(ROTATED_HEXAGON_H0_KEY, ROTATED_HEXAGON_H0_SCALE)

# fill_domain_mesh is the generic path: pass any domain oracle and get a
# PrismMesh back for plotting or MicromagProblem setup.
sphere = fill_domain_mesh(
    sphere_domain,
    N_target=TARGET_TILES,
    h0=sphere_h0,
    h_min=sphere_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
ellipsoid = fill_domain_mesh(
    ellipsoid_domain,
    N_target=TARGET_TILES,
    h0=ellipsoid_h0,
    h_min=ellipsoid_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
cylinder = fill_domain_mesh(
    cylinder_domain,
    N_target=TARGET_TILES,
    h0=cylinder_h0,
    h_min=cylinder_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
hexagonal_prism = fill_domain_mesh(
    hexagon_domain,
    N_target=TARGET_TILES,
    h0=hexagon_h0,
    h_min=hexagon_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
rotated_hexagonal_prism = fill_domain_mesh(
    rotated_hexagon_domain,
    N_target=TARGET_TILES,
    h0=rotated_hexagon_h0,
    h_min=rotated_hexagon_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)

# Compatibility wrappers are shorter calls for common analytic shapes. They
# construct the domain oracle internally and then call fill_domain_mesh.
sphere_from_wrapper = generate_sphere(
    SPHERE_RADIUS,
    TARGET_TILES,
    h0=sphere_h0,
    h_min=sphere_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
ellipsoid_from_wrapper = generate_ellipsoid(
    ELLIPSOID_SEMI_AXES,
    TARGET_TILES,
    h0=ellipsoid_h0,
    h_min=ellipsoid_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
cylinder_from_wrapper = generate_cylinder(
    CYLINDER_RADIUS,
    CYLINDER_LENGTH,
    CYLINDER_AXIS,
    TARGET_TILES,
    h0=cylinder_h0,
    h_min=cylinder_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
hexagon_from_wrapper = generate_hexagonal_prism(
    HEXAGON_SIDE_LENGTH,
    HEXAGON_HEIGHT,
    HEXAGON_AXIS,
    TARGET_TILES,
    rotation_degrees=HEXAGON_ROTATION_DEGREES,
    h0=hexagon_h0,
    h_min=hexagon_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
rotated_hexagon_from_wrapper = generate_hexagonal_prism(
    HEXAGON_SIDE_LENGTH,
    HEXAGON_HEIGHT,
    HEXAGON_AXIS,
    TARGET_TILES,
    rotation_degrees=ROTATED_HEXAGON_DEGREES,
    h0=rotated_hexagon_h0,
    h_min=rotated_hexagon_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)

meshes = {
    'Rectangular prism': box,
    'Sphere': sphere,
    'Ellipsoid': ellipsoid,
    'Cylinder': cylinder,
    'Hexagonal prism': hexagonal_prism,
    'Rotated hexagon': rotated_hexagonal_prism,
}
adaptive_meshes = {
    'Sphere': (sphere, sphere_domain),
    'Ellipsoid': (ellipsoid, ellipsoid_domain),
    'Cylinder': (cylinder, cylinder_domain),
    'Hexagonal prism': (hexagonal_prism, hexagon_domain),
    'Rotated hexagon': (rotated_hexagonal_prism, rotated_hexagon_domain),
}

print(
    f"{'Shape':<22} {'Cells':>8} {'Levels':>10} {'h0 [nm]':>10} "
    f"{'min h [nm]':>12} {'Unresolved':>11} {'Fill':>10}  Target status"
)
print('-' * 118)
for name, mesh in meshes.items():
    levels = f'{mesh.levels.min()}-{mesh.levels.max()}'
    stats = mesh.refinement_metadata.get('cube_fill_stats', {})
    h0_nm = '-' if 'h0' not in stats else f"{stats['h0'] / NM:.2f}"
    min_h_nm = f'{np.min(mesh.dimensions[:, 0]) / NM:.2f}'
    unresolved = '-' if 'boundary_cells_left_unresolved' not in stats else stats['boundary_cells_left_unresolved']
    fill = '-' if stats.get('fill_fraction') is None else f"{stats['fill_fraction']:.1%}"
    stop_reason = stats.get('target_status', mesh.refinement_metadata.get('termination_reason', '-'))
    print(
        f'{name:<22} {mesh.achieved_tiles:>8} {levels:>10} {h0_nm:>10} '
        f'{min_h_nm:>12} {unresolved:>11} {fill:>10}  {stop_reason}'
    )
    for diagnostic in mesh.diagnostics:
        print(f'  Note: {diagnostic}')


## Automatic checks

These checks validate array shapes, positive dimensions, root bounds, deterministic wrappers, conservative containment, and non-overlap. An assertion failure points to a mesh-generation problem.


In [ ]:
def validate_mesh(mesh):
    assert mesh.centers.shape == (mesh.achieved_tiles, 3)
    assert mesh.dimensions.shape == mesh.centers.shape
    assert mesh.levels.shape == (mesh.achieved_tiles,)
    assert np.all(mesh.dimensions > 0)
    lower = mesh.centers - mesh.dimensions / 2
    upper = mesh.centers + mesh.dimensions / 2
    tolerance = max(float(np.max(mesh.root_bounds[1] - mesh.root_bounds[0])) * 1e-12, 1e-30)
    assert np.all(lower >= mesh.root_bounds[0] - tolerance)
    assert np.all(upper <= mesh.root_bounds[1] + tolerance)
    unique = np.unique(np.column_stack((mesh.centers, mesh.levels)), axis=0)
    assert len(unique) == mesh.achieved_tiles

for mesh in meshes.values():
    validate_mesh(mesh)

assert box.achieved_tiles == math.prod(BOX_RESOLUTION)
assert np.all(box.levels == 0)
np.testing.assert_array_equal(sphere.centers, sphere_from_wrapper.centers)
np.testing.assert_array_equal(sphere.dimensions, sphere_from_wrapper.dimensions)
np.testing.assert_array_equal(sphere.levels, sphere_from_wrapper.levels)
np.testing.assert_array_equal(ellipsoid.centers, ellipsoid_from_wrapper.centers)
np.testing.assert_array_equal(ellipsoid.dimensions, ellipsoid_from_wrapper.dimensions)
np.testing.assert_array_equal(ellipsoid.levels, ellipsoid_from_wrapper.levels)
np.testing.assert_array_equal(cylinder.centers, cylinder_from_wrapper.centers)
np.testing.assert_array_equal(cylinder.dimensions, cylinder_from_wrapper.dimensions)
np.testing.assert_array_equal(cylinder.levels, cylinder_from_wrapper.levels)
np.testing.assert_array_equal(hexagonal_prism.centers, hexagon_from_wrapper.centers)
np.testing.assert_array_equal(hexagonal_prism.dimensions, hexagon_from_wrapper.dimensions)
np.testing.assert_array_equal(hexagonal_prism.levels, hexagon_from_wrapper.levels)
np.testing.assert_array_equal(
    rotated_hexagonal_prism.centers,
    rotated_hexagon_from_wrapper.centers,
)
np.testing.assert_array_equal(
    rotated_hexagonal_prism.dimensions,
    rotated_hexagon_from_wrapper.dimensions,
)
np.testing.assert_array_equal(
    rotated_hexagonal_prism.levels,
    rotated_hexagon_from_wrapper.levels,
)
single_level_meshes = [
    name
    for name, (mesh, _domain) in adaptive_meshes.items()
    if np.min(mesh.levels) == np.max(mesh.levels)
]
if single_level_meshes:
    print(
        'Single-level complete-layer meshes:', single_level_meshes,
        '- this is valid for calibrated h0 values; choose a different calibrated TARGET_TILES value if you want more visible refinement.'
    )

for name, (mesh, domain) in adaptive_meshes.items():
    stats = mesh.refinement_metadata['cube_fill_stats']
    assert stats['accepted_count'] == mesh.achieved_tiles
    assert stats['classified_cells'] >= mesh.achieved_tiles
    assert mesh.achieved_tiles > 0
    cubes = [
        type('CubeLike', (), {
            'center': center,
            'h': float(dimensions[0]),
            'level': int(level),
            'i': 0,
            'j': 0,
            'k': 0,
        })()
        for center, dimensions, level in zip(mesh.centers, mesh.dimensions, mesh.levels)
    ]
    for cube in cubes:
        state = domain.classify_cube(cube, EPS)
        assert state.name == 'INSIDE', f'{name} accepted a non-inside cube.'
    assert_no_cube_overlaps(cubes, tol=1e-12)

print('All conservative mesh checks passed.')


## Verify conservative containment

For sphere domains, accepted cubes must satisfy the conservative SDF criterion `sdf(center) >= sqrt(3) * h / 2 + eps`. Box, ellipsoid, cylinder, and hexagonal prism checks use their exact/conservative interval classifiers, because those can safely accept cells that the generic SDF ball test would classify as ambiguous.


In [ ]:
for center, dimensions in zip(sphere.centers, sphere.dimensions):
    h = float(dimensions[0])
    required = sqrt(3.0) * h / 2.0 + EPS
    assert sphere_domain.sdf(center) >= required

box_cubes, box_stats = adaptive_cube_fill(
    domain=box_domain,
    bbox=box_domain.bounding_box,
    N_target=10_000,
    h0=10 * NM,
    h_min=10 * NM,
    max_depth=0,
    eps=EPS,
    overshoot_policy='soft',
    queue_policy=QUEUE_POLICY,
)
for cube in box_cubes:
    assert box_domain.classify_cube(cube, EPS).name == 'INSIDE'
assert math.isclose(box_stats.accepted_volume, np.prod(BOX_DIMENSIONS), rel_tol=1e-12)
assert math.isclose(box_stats.fill_fraction, 1.0, rel_tol=1e-12)

print('Conservative containment checks passed.')


## Refinement diagnostics

The conservative filler refines ambiguous boundary cells breadth-first by default. Smaller dyadic cubes should appear near curved boundaries, and unresolved boundary cells are reported instead of being accepted.


In [ ]:
print(f"{'Shape':<18} {'Cells':>7} {'Classified':>11} {'Outside':>9} {'Unresolved':>11} {'Max lvl':>8}  Stop reason")
print('-' * 104)
for name, (mesh, _domain) in adaptive_meshes.items():
    stats = mesh.refinement_metadata['cube_fill_stats']
    print(
        f"{name:<18} {mesh.achieved_tiles:>7} "
        f"{stats['classified_cells']:>11} "
        f"{stats['rejected_outside_cells']:>9} "
        f"{stats['boundary_cells_left_unresolved']:>11} "
        f"{stats['max_level_reached']:>8}  "
        f"{stats['termination_reason']}"
    )

fig, axes = plt.subplots(1, len(adaptive_meshes), figsize=(10, 4), sharey=True)
if len(adaptive_meshes) == 1:
    axes = [axes]
for ax, (name, (mesh, _domain)) in zip(axes, adaptive_meshes.items()):
    level_counts = mesh.refinement_metadata['level_counts']
    levels = np.array(sorted(level_counts))
    counts = np.array([level_counts[level] for level in levels])
    ax.bar(levels, counts)
    ax.set_xticks(levels)
    ax.set_title(name)
    ax.set_xlabel('Refinement level')
axes[0].set_ylabel('Number of cells')
fig.tight_layout()


## Determinism and target-policy checks


In [ ]:
sphere_reference = generate_sphere(
    SPHERE_RADIUS,
    TARGET_TILES,
    h0=sphere_h0,
    h_min=sphere_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
sphere_repeat = generate_sphere(
    SPHERE_RADIUS,
    TARGET_TILES,
    h0=sphere_h0,
    h_min=sphere_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    queue_policy=QUEUE_POLICY,
)
np.testing.assert_array_equal(sphere_reference.centers, sphere_repeat.centers)
np.testing.assert_array_equal(sphere_reference.dimensions, sphere_repeat.dimensions)
np.testing.assert_array_equal(sphere_reference.levels, sphere_repeat.levels)
assert sphere_reference.diagnostics == sphere_repeat.diagnostics

POLICY_TARGET = 2**11
policy_h0 = coarse_h0_for_target(SPHERE_H0_KEY, SPHERE_H0_SCALE, POLICY_TARGET)
policy_h_min = h_min_for_target(SPHERE_H0_KEY, SPHERE_H0_SCALE, POLICY_TARGET)

never_exceed_sphere = generate_sphere(
    SPHERE_RADIUS,
    POLICY_TARGET,
    h0=policy_h0,
    h_min=policy_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    overshoot_policy='never_exceed',
    queue_policy=QUEUE_POLICY,
    complete_layers=False,
)
assert never_exceed_sphere.achieved_tiles <= POLICY_TARGET
assert never_exceed_sphere.target_reached

soft_sphere = generate_sphere(
    SPHERE_RADIUS,
    POLICY_TARGET,
    h0=policy_h0,
    h_min=policy_h_min,
    max_depth=MAX_DEPTH,
    eps=EPS,
    overshoot_policy='soft',
    queue_policy=QUEUE_POLICY,
    complete_layers=False,
)
assert soft_sphere.achieved_tiles >= POLICY_TARGET or soft_sphere.refinement_metadata['cube_fill_stats']['boundary_cells_left_unresolved'] > 0

try:
    generate_sphere(SPHERE_RADIUS, TARGET_TILES, 2, 5)
except TypeError:
    pass
else:
    raise AssertionError('Old generate_sphere(radius, target, min, max) call still works.')

try:
    coarse_h0_for_target(SPHERE_H0_KEY, SPHERE_H0_SCALE, 2**13)
except ValueError as exc:
    assert CALIBRATION_ERROR in str(exc)
else:
    raise AssertionError('Unsupported calibrated target did not raise ValueError.')

print('Determinism, legacy target-policy, and calibrated-target checks passed.')


## Compare the generated shapes

Cells are colored by refinement level. For very large meshes, `max_plot_tiles` limits rendering cost without changing the generated mesh.


In [ ]:
fig = plt.figure(figsize=(15, 9))
for index, (name, mesh) in enumerate(meshes.items(), start=1):
    ax = fig.add_subplot(2, 3, index, projection='3d')
    plot_prism_mesh(
        mesh,
        fig=fig,
        ax=ax,
        alpha=0.45,
        max_plot_tiles=6000,
        elevation=25,
        azimuth=35,
    )
    ax.set_title(f'{name}: {mesh.achieved_tiles} cells')
fig.tight_layout()


## Volume covered by each refinement level

A high refinement level can contain many cells while covering only a small part of the accepted volume.


In [ ]:
fig, axes = plt.subplots(len(adaptive_meshes), 2, figsize=(10, 3 * len(adaptive_meshes)), sharex=False)

for row, (name, (mesh, _domain)) in enumerate(adaptive_meshes.items()):
    levels = np.unique(mesh.levels)
    counts = np.array([np.count_nonzero(mesh.levels == level) for level in levels])
    cell_volumes = np.prod(mesh.dimensions, axis=1)
    volumes = np.array([cell_volumes[mesh.levels == level].sum() for level in levels])
    count_fraction = 100 * counts / counts.sum()
    volume_fraction = 100 * volumes / volumes.sum()

    axes[row, 0].bar(levels, count_fraction)
    axes[row, 1].bar(levels, volume_fraction, color='tab:orange')
    axes[row, 0].set_ylabel(f'{name}\nShare [%]')
    axes[row, 0].set_xticks(levels)
    axes[row, 1].set_xticks(levels)
    axes[row, 0].grid(axis='y', alpha=0.3)
    axes[row, 1].grid(axis='y', alpha=0.3)

axes[0, 0].set_title('Fraction of cells')
axes[0, 1].set_title('Fraction of accepted volume')
axes[-1, 0].set_xlabel('Refinement level')
axes[-1, 1].set_xlabel('Refinement level')
fig.tight_layout()


## Coarse-resolution calibration reference

The starting coarse size `h0` strongly affects adaptive fills. The analytic estimate `estimate_h0_from_volume(domain, target)` gives a scale from volume and target cell count. The reusable lookup table `H0_REFERENCE_TABLE` stores dimensionless multipliers around that estimate. `Target min` and `Target max` in that table are lookup ranges for `N_target`, not generated tile counts.

The sweep below tests several multipliers and target counts. Tiny `h0` values can create far too many cells, so selected rows must first be within the target-count band. Among target-valid rows, selection maximizes conservative fill fraction, then uses target-count error, unresolved boundary cells, and classified-cell count as tie-breaks. The result is a reference table that any notebook or future script can reproduce with `calibrate_h0_sweep(...)` and `select_best_h0_records(...)`.


In [ ]:
def print_records_table(records, columns, title=None):
    if title:
        print(title)
    if not records:
        print('(no records)')
        return
    text_rows = []
    for record in records:
        text_row = []
        for key, header, formatter in columns:
            value = record.get(key)
            text_row.append(str(formatter(value)) if formatter else str(value))
        text_rows.append(text_row)
    widths = [len(header) for _key, header, _formatter in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    header = '  '.join(header.ljust(width) for width, (_key, header, _formatter) in zip(widths, columns))
    print(header)
    print('-' * len(header))
    for row in text_rows:
        print('  '.join(value.ljust(width) for value, width in zip(row, widths)))

lookup_rows = []
for shape_key, target_map in CALIBRATED_RELATIVE_H0.items():
    for target, h0_relative in target_map.items():
        lookup_rows.append({
            'shape_key': shape_key,
            'target': int(target),
            'h0_relative': h0_relative,
        })
lookup_rows.sort(key=lambda row: (row['shape_key'], row['target']))

lookup_columns = [
    ('shape_key', 'Shape key', None),
    ('target', 'Target', lambda value: f'{value:d}'),
    ('h0_relative', 'h0 / scale', lambda value: f'{value:.6f}'),
]
print_records_table(lookup_rows, lookup_columns, 'Calibrated relative h0 lookup')

print()
print(f'Sphere calibrated h0 for TARGET_TILES={TARGET_TILES}: {sphere_h0 / NM:.2f} nm')
print(f'Sphere relative h0: {calibrated_relative_h0(SPHERE_H0_KEY, TARGET_TILES):.6f}')


## Conservative fill convergence

Accepted volume approaches the analytic domain volume from below as target count and refinement depth increase. Some volume can remain unfilled near boundaries because ambiguous cubes are never accepted.


In [ ]:
target_values = np.array(CALIBRATED_TARGET_TILES)
convergence_cases = {
    'Sphere': (
        lambda target: generate_sphere(
            SPHERE_RADIUS,
            int(target),
            h0=coarse_h0_for_target(SPHERE_H0_KEY, SPHERE_H0_SCALE, target),
            h_min=h_min_for_target(SPHERE_H0_KEY, SPHERE_H0_SCALE, target),
            max_depth=MAX_DEPTH,
            eps=EPS,
            queue_policy=QUEUE_POLICY,
        ),
        sphere_domain.volume,
    ),
    'Ellipsoid': (
        lambda target: generate_ellipsoid(
            ELLIPSOID_SEMI_AXES,
            int(target),
            h0=coarse_h0_for_target(ELLIPSOID_H0_KEY, ELLIPSOID_H0_SCALE, target),
            h_min=h_min_for_target(ELLIPSOID_H0_KEY, ELLIPSOID_H0_SCALE, target),
            max_depth=MAX_DEPTH,
            eps=EPS,
            queue_policy=QUEUE_POLICY,
        ),
        ellipsoid_domain.volume,
    ),
    'Cylinder': (
        lambda target: generate_cylinder(
            CYLINDER_RADIUS,
            CYLINDER_LENGTH,
            CYLINDER_AXIS,
            int(target),
            h0=coarse_h0_for_target(CYLINDER_H0_KEY, CYLINDER_H0_SCALE, target),
            h_min=h_min_for_target(CYLINDER_H0_KEY, CYLINDER_H0_SCALE, target),
            max_depth=MAX_DEPTH,
            eps=EPS,
            queue_policy=QUEUE_POLICY,
        ),
        cylinder_domain.volume,
    ),
    'Hexagonal prism': (
        lambda target: generate_hexagonal_prism(
            HEXAGON_SIDE_LENGTH,
            HEXAGON_HEIGHT,
            HEXAGON_AXIS,
            int(target),
            rotation_degrees=HEXAGON_ROTATION_DEGREES,
            h0=coarse_h0_for_target(HEXAGON_H0_KEY, HEXAGON_H0_SCALE, target),
            h_min=h_min_for_target(HEXAGON_H0_KEY, HEXAGON_H0_SCALE, target),
            max_depth=MAX_DEPTH,
            eps=EPS,
            queue_policy=QUEUE_POLICY,
        ),
        hexagon_domain.volume,
    ),
    'Rotated hexagon': (
        lambda target: generate_hexagonal_prism(
            HEXAGON_SIDE_LENGTH,
            HEXAGON_HEIGHT,
            HEXAGON_AXIS,
            int(target),
            rotation_degrees=ROTATED_HEXAGON_DEGREES,
            h0=coarse_h0_for_target(ROTATED_HEXAGON_H0_KEY, ROTATED_HEXAGON_H0_SCALE, target),
            h_min=h_min_for_target(ROTATED_HEXAGON_H0_KEY, ROTATED_HEXAGON_H0_SCALE, target),
            max_depth=MAX_DEPTH,
            eps=EPS,
            queue_policy=QUEUE_POLICY,
        ),
        rotated_hexagon_domain.volume,
    ),
}

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, (generator, analytic_volume) in convergence_cases.items():
    refined_meshes = [generator(target) for target in target_values]
    accepted_volumes = np.array([mesh.represented_volume for mesh in refined_meshes])
    fill_fractions = accepted_volumes / analytic_volume
    ax.plot(target_values, fill_fractions, marker='o', label=name)

ax.axhline(1.0, color='black', linewidth=1, linestyle='--')
ax.set_xscale('log', base=2)
ax.set_xlabel('Target cubes')
ax.set_ylabel('Accepted volume / analytic volume')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()


## Verify MagTense compatibility

This constructs a `MicromagProblem` from the conservative sphere mesh but does not run the solver.


In [ ]:
from magtense.micromag import MicromagProblem

grid = sphere.to_micromag_kwargs()
problem = MicromagProblem(
    **grid,
    grid_L=list(sphere.root_bounds[1] - sphere.root_bounds[0]),
    m0=np.tile([0.0, 0.0, 1.0], (sphere.achieved_tiles, 1)),
)
assert problem.ntot == sphere.achieved_tiles
assert problem.grid_type == 3
np.testing.assert_array_equal(problem.grid_pts, sphere.centers)
np.testing.assert_array_equal(problem.grid_abc, sphere.dimensions)
print(f'MicromagProblem constructed with {problem.ntot} unstructured prism cells.')


## Interactive rotation

Run `%matplotlib widget` before this cell when `ipympl` is installed to enable mouse rotation as well as the elevation and azimuth sliders. With the normal inline backend, the sliders still update the figure in notebook frontends that redraw Matplotlib canvases.


In [ ]:
fig, ax, rotation_controls = interactive_prism_mesh(
    rotated_hexagonal_prism,
    max_plot_tiles=6000,
    alpha=0.45,
    elevation=35,
)
display(rotation_controls)
